# SRC-001 PCAP Audit

## Purpose

Audit the SRC-001 packet captures to determine their readability, metadata,
traffic structure, temporal coverage, and usefulness as packet-level evidence
for the DAICP MVP.

## Out of Scope

This notebook does not:

- modify packet captures;
- perform deep packet forensics;
- train intrusion-detection models;
- perform cross-source alignment;
- evaluate RCA performance.

## 1. Setup

Locate the SRC-001 PCAP captures and verify whether packet-analysis utilities
are available in the current execution environment.

The raw captures are discovered dynamically from the SRC-001 source directory.
No packet data is loaded into memory at this stage.

In [1]:
import subprocess
from io import StringIO

import pandas as pd

from src.data.inventory import DataInventory
from src.data.dataset_tools import PacketTools

In [2]:
SOURCE_ID = "SRC-001"

inventory = DataInventory(source_id=SOURCE_ID)

RAW_DATA_PATH = inventory.raw_data_path
packet_tools = PacketTools("capinfos", "tshark").get_packet_tools()

In [3]:
pcap_files = sorted(
    path
    for path in RAW_DATA_PATH.iterdir()
    if path.is_file() and path.suffix.lower() == ".pcap"
)

if not pcap_files:
    raise FileNotFoundError(
        f"No PCAP files found in: {RAW_DATA_PATH}"
    )

pcap_files

[PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/benign.pcap'),
 PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/ddos_icmp_hping3.pcap'),
 PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/ftp_hydra.pcap'),
 PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/icmp_nping.pcap'),
 PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/portscan_masscan.pcap'),
 PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/web_xss.pcap')]

## 2. Capture Inventory

**Objective:** Identify the available PCAP files and their sizes.

**Method:** Dynamically discover PCAP files from the SRC-001 raw
data directory without loading packet contents into memory.

In [4]:
pcap_files = sorted(
    path for path in RAW_DATA_PATH.iterdir()
    if path.is_file() and path.suffix.lower() == ".pcap"
)

if not pcap_files:
    raise FileNotFoundError(f"No PCAP files found in {RAW_DATA_PATH}")

capture_inventory = pd.DataFrame([
    {
        "scenario": path.stem,
        "file_name": path.name,
        "size_mb": round(path.stat().st_size / (1024 ** 2), 2)
    }
    for path in pcap_files
])

display(capture_inventory)

,scenario,file_name,size_mb
0,benign,benign.pcap,67.81
1,ddos_icmp_hping3,ddos_icmp_hping3.pcap,213.82
2,ftp_hydra,ftp_hydra.pcap,384.83
3,icmp_nping,icmp_nping.pcap,481.71
4,portscan_masscan,portscan_masscan.pcap,833.01
5,web_xss,web_xss.pcap,134.01


## 3. Capture Validation

**Objective:** Verify the readability and structural characteristics of the selected PCAP files.

**Method:** Use `capinfos` to extract capture format, encapsulation, packet count, file size, duration, and chronological consistency.

**Scope:** Structural validation only. Packet-level forensic analysis is excluded.

In [5]:
result = subprocess.run(
    [
        packet_tools["capinfos"],
        "-C", "-T", "-t", "-E", "-c", "-s", "-u", "-o",
        *[str(path) for path in pcap_files]
    ],
    capture_output=True, text=True, check=True
)

capture_metadata = pd.read_csv(StringIO(result.stdout), sep="\t")

display(capture_metadata)


,File name,File type,File encapsulation,Number of packets,File size (bytes),Capture duration (seconds),Strict time order
0,/Users/emiliogarcialopez/projectGitHub/distrib...,pcap,linux-sll,140319,71108907,96.382330,False
1,File name,File type,File encapsulation,Number of packets,File size (bytes),Capture duration (seconds),Strict time order
2,/Users/emiliogarcialopez/projectGitHub/distrib...,pcap,linux-sll,409452,224211674,258.487075,False
3,File name,File type,File encapsulation,Number of packets,File size (bytes),Capture duration (seconds),Strict time order
4,/Users/emiliogarcialopez/projectGitHub/distrib...,pcap,linux-sll,1391246,403521743,836.377660,False
5,File name,File type,File encapsulation,Number of packets,File size (bytes),Capture duration (seconds),Strict time order
6,/Users/emiliogarcialopez/projectGitHub/distrib...,pcap,linux-sll,5657405,505113905,606.373024,False
7,File name,File type,File encapsulation,Number of packets,File size (bytes),Capture duration (seconds),Strict time order
8,/Users/emiliogarcialopez/projectGitHub/distrib...,pcap,linux-sll,6605143,873475951,610.058419,False
9,File name,File type,File encapsulation,Number of packets,File size (bytes),Capture duration (seconds),Strict time order


### 3.1. Structural Validation Results

**Objective:** Determine whether the selected PCAP files meet the minimum structural requirements for further analysis.

**Validation criteria:**
- The capture format is readable by `capinfos`.
- The capture contains at least one packet.
- The capture duration is valid and non-negative.

Passing these checks does not guarantee that all packets are complete or correctly decoded.

In [6]:
capture_metadata = capture_metadata.loc[
    capture_metadata["File name"].ne("File name")
].copy()

capture_metadata["Number of packets"] = pd.to_numeric(capture_metadata["Number of packets"], errors="raise")
capture_metadata["Capture duration (seconds)"] = pd.to_numeric(capture_metadata["Capture duration (seconds)"], errors="raise")

capture_metadata["valid_structure"] = (
    capture_metadata["Number of packets"].gt(0)
    & capture_metadata["Capture duration (seconds)"].ge(0)
)

validation_summary = capture_metadata[
    ["File name", "Number of packets", "Capture duration (seconds)", "valid_structure"]
].reset_index(drop=True)

assert len(validation_summary) == len(pcap_files), "Capture count mismatch"

display(validation_summary)
print(f"Valid captures: {validation_summary['valid_structure'].sum()}/{len(validation_summary)}")

,File name,Number of packets,Capture duration (seconds),valid_structure
0,/Users/emiliogarcialopez/projectGitHub/distrib...,140319,96.382330,True
1,/Users/emiliogarcialopez/projectGitHub/distrib...,409452,258.487075,True
2,/Users/emiliogarcialopez/projectGitHub/distrib...,1391246,836.377660,True
3,/Users/emiliogarcialopez/projectGitHub/distrib...,5657405,606.373024,True
4,/Users/emiliogarcialopez/projectGitHub/distrib...,6605143,610.058419,True
5,/Users/emiliogarcialopez/projectGitHub/distrib...,440052,243.840014,True


Valid captures: 6/6


### Finding — PCAP Structural Audit

Six PCAP captures were identified and examined using Wireshark command-line utilities.

All six captures passed the basic structural validation criteria, including successful metadata extraction, non-empty packet content, and valid capture duration.

A preliminary inspection of 1,000 packets per capture identified differences in protocol composition across scenarios.

Wireshark also reported decoding anomalies during the full inspection of `benign.pcap`. Their causes remain unverified.

**Decision:** The six captures are suitable for further dataset evaluation at the structural level. Their suitability as Root Cause Analysis evidence remains subject to cross-source consistency and evaluation-risk assessment.

## 4. Protocol Overview

**Objective:** Identify the network protocols visible in the selected PCAP captures and detect preliminary decoding anomalies.

**Method:** Use TShark to inspect the first 1,000 packets of each capture and identify relevant protocol layers.

**Scope:** Exploratory inspection only. The sample is not necessarily representative of the complete capture, and protocols absent from the sample may still appear elsewhere in the file.

In [7]:
protocols = ["ip", "tcp", "udp", "sctp", "icmp", "gtp", "x2ap", "ngap"]
sample_size = 1000
rows = []

for path in pcap_files:
    result = subprocess.run(
        [packet_tools["tshark"], "-r", str(path), "-c", str(sample_size), "-T", "fields", "-e", "frame.protocols"],
        capture_output=True, text=True, check=True
    )

    layers = [set(line.split(":")) for line in result.stdout.splitlines()]
    counts = {proto: sum(proto in packet for packet in layers) for proto in protocols}

    rows.append({
        "scenario": path.stem,
        "sampled_packets": len(layers),
        **counts,
        "malformed": sum("_ws.malformed" in packet for packet in layers)
    })

protocol_summary = pd.DataFrame(rows)
display(protocol_summary)


,scenario,sampled_packets,ip,tcp,udp,sctp,icmp,gtp,x2ap,ngap,malformed
0,benign,1000,996,346,234,452,0,150,149,0,0
1,ddos_icmp_hping3,1000,995,463,150,479,0,144,159,0,0
2,ftp_hydra,1000,995,240,92,723,0,90,240,0,0
3,icmp_nping,1000,1000,16,300,100,784,300,33,0,0
4,portscan_masscan,1000,1000,748,433,108,0,433,36,0,0
5,web_xss,1000,997,121,39,863,0,39,287,0,0


## 5. Audit Findings

### 5.1. Structural Integrity

The six selected PCAP captures were inventoried and examined using `capinfos`.

Structural validation considers file readability, packet availability, and valid capture duration.

The validation results determine which captures can proceed to subsequent analysis.

### 5.2. Protocol Inspection

A preliminary protocol inspection was performed using TShark, examining up to 1,000 initial packets per capture.

The inspection identifies visible network protocols and potential decoding anomalies. These results are exploratory and do not represent a complete characterization of each capture.

### 5.3. Identified Limitations

- Original capture conditions and packet-loss information are not fully documented.
- Protocol inspection is limited to an initial packet sample.
- Wireshark identified decoding anomalies in `benign.pcap`, including X2AP malformed-packet indications. Their underlying cause remains unverified.
- Structural validity does not guarantee correctness of scenario labels or suitability for RCA.

### 5.4. Audit Decision

Captures passing structural validation are candidates for further evaluation.

Final dataset suitability remains subject to cross-source consistency checks, scenario relevance, and evaluation-risk assessment.

In [8]:
print("=== PCAP Audit Summary ===")
print(f"Captures inspected: {len(validation_summary)}")
print(f"Structurally valid: {validation_summary['valid_structure'].sum()}")
print(f"Structural failures: {(~validation_summary['valid_structure']).sum()}")

display(validation_summary)
display(protocol_summary)


=== PCAP Audit Summary ===
Captures inspected: 6
Structurally valid: 6
Structural failures: 0


,File name,Number of packets,Capture duration (seconds),valid_structure
0,/Users/emiliogarcialopez/projectGitHub/distrib...,140319,96.382330,True
1,/Users/emiliogarcialopez/projectGitHub/distrib...,409452,258.487075,True
2,/Users/emiliogarcialopez/projectGitHub/distrib...,1391246,836.377660,True
3,/Users/emiliogarcialopez/projectGitHub/distrib...,5657405,606.373024,True
4,/Users/emiliogarcialopez/projectGitHub/distrib...,6605143,610.058419,True
5,/Users/emiliogarcialopez/projectGitHub/distrib...,440052,243.840014,True


,scenario,sampled_packets,ip,tcp,udp,sctp,icmp,gtp,x2ap,ngap,malformed
0,benign,1000,996,346,234,452,0,150,149,0,0
1,ddos_icmp_hping3,1000,995,463,150,479,0,144,159,0,0
2,ftp_hydra,1000,995,240,92,723,0,90,240,0,0
3,icmp_nping,1000,1000,16,300,100,784,300,33,0,0
4,portscan_masscan,1000,1000,748,433,108,0,433,36,0,0
5,web_xss,1000,997,121,39,863,0,39,287,0,0


### 5.4. Conclusion and Dataset Suitability

The six selected PCAP files passed the basic structural validation checks. All captures are readable, contain packet-level data, and provide valid temporal metadata.

Preliminary protocol inspection confirms the presence of network traffic that can support further technical investigation. However, the inspection was limited to 1,000 packets per capture and does not establish complete protocol coverage or scenario correctness.

Decoding anomalies identified in `benign.pcap` remain unresolved and must be considered when interpreting packet-level evidence.

**Decision: Conditionally Suitable**

The PCAP files are suitable for further processing and evaluation within the DAICP MVP. However, their suitability as evidence for Root Cause Analysis has not yet been established.

Further validation is required to determine cross-source consistency, scenario relevance, and potential evaluation risks.

**Next step:** Proceed to cross-source consistency analysis between PCAP captures and TXT telemetry.